In [1]:
import time
from pathlib import Path

from src.utils import parse_toml, utils
from src.utils.batch import generate_processing_batch
from src.preprocessing import preprocess, cross_sections, network_smoothing
from src.metrics import channel_cross_section_metrics as channel_metrics
from src.metrics import channel_curvature_metrics as curvature_metrics
from src.metrics import flood_inundation_map as fim
from src.metrics import floodplain_metrics

from src.postprocessing import spatial_qc as qc

In [ ]:
# Debug WBT compile issue only on WSL Ubuntu 20.0
# uncomment and run when whitebox install breaks
# whitebox.download_wbt(linux_musl=True, reset=True)

### Define config, set filepaths, logging and ready to run!

In [ ]:
config_toml = Path("src/config.toml")
fpaths_toml = Path("src/utils/filepaths.toml") # no need to change or modify

# step 1
Config = parse_toml.create_config(config_toml)

# step 2
hucs = ['01234567890'] # single huc
# hucs = generate_processing_batch(Config.batch_csv) # or pass a batch

# step 3
Paths = parse_toml.create_filepaths(fpaths_toml, Config, huc)
utils.create_folder(Paths)

# logging
logger = utils.initialize_logger(Paths.log)

# log input parameters
Paths_dict = parse_toml.class_to_dict(Paths)
for k,v in Paths_dict.items():
    logger.debug(f"{k}: {v}")

### Preprocess

In [ ]:
preprocess.run_preprocessing_steps(Config, Paths, logger)

### Generate cross-sections

In [ ]:
cross_sections.generate(Config, Paths)

### 1D Channel Cross-section Metrics


In [ ]:
channel_metrics.derive(
    Config.methods['cross_section']['cell_size'],
    Paths.elevation_profiles,
    Paths.channel_xns,
    Paths.dem,
    Paths.bank_points,
    Config.methods['cross_section'],
    Config.spatial_ref['epsg'],
    logger
    )

### Channel Curvature Metrics

In [ ]:
curvature_metrics.derive(
    Paths.xn_coordinates,
    Paths.dem,
    Paths.bank_pixels,
    Config.spatial_ref['cell_size'],
    Config.methods['curvature'],
    Paths.network_poly,
    Paths.channel_segs,
    logger
    )

### Delineate flood inundation layer

In [ ]:
reach_id = Config.preprocess['reach-order']['reach_id']
min_da, max_da = Config.methods['flood_thresholds'].values()
fim.delineate(
    Paths.hand,
    Paths.sub_watersheds_poly,
    Config.preprocess['reach-order']['reach_id'],
    Paths.flood_extent_layer,
    Paths.flood_height_thresholds,
    min_da,
    max_da,
    logger
    )

### 1D Floodplain Cross-section Metrics

In [ ]:
floodplain_metrics.derive(
    Paths.floodplain_xns, Paths.flood_extent_layer,
    Paths.dem, "CURVE_WD",
    Paths.channel_segs, Config.xn_lengths["floodplain"],
    logger
)

### Experimental HAND method

In [ ]:
floodplain_metrics.hand_method(
    Paths.hand, Paths.channel_segs, Paths.network_poly,
    Paths.network_rast, Paths.flood_extent_layer, Paths.dem,
    Config.xn_lengths["floodplain"], logger
)

### Quality checks

In [ ]:
flowline_mask = qc.create_flowline_qc_mask(
    Paths.flowlines, 
    Config.postprocess['stream-buffer'],
    Paths.watershed
)

waterbody_mask = qc.create_waterbody_qc_mask(
    Config.ancillary['nhd_wbds'],
    [390, 436], 
    Paths.watershed
    )

# flag bankpoints:
bank_points_qc = utils.vector_to_geodataframe(Paths.bank_points)
bank_points_qc = qc.flag_features_by_qc_mask(
    bank_points_qc, flowline_mask, "NHD_Flag", "xn_num"
    )
bank_points_qc = qc.flag_features_by_qc_mask(
    bank_points_qc, waterbody_mask, "WBD_Flag", "xn_num", output=Paths.bank_points
    )

# flag channel segs:
channel_segs_qc = utils.vector_to_geodataframe(Paths.channel_segs)

channel_segs_qc = qc.flag_features_by_qc_mask(
    channel_segs_qc, flowline_mask, "NHD_Flag"
    )
channel_segs_qc = qc.flag_features_by_qc_mask(
    channel_segs_qc, waterbody_mask, "WBD_Flag", output=Paths.channel_segs
    )

# flag floodplain xns:
floodplain_xns_qc = utils.vector_to_geodataframe(Paths.floodplain_xns)

floodplain_xns_qc = qc.flag_features_by_qc_mask(
    floodplain_xns_qc, flowline_mask, "NHD_Flag"
    )
floodplain_xns_qc = qc.flag_features_by_qc_mask(
    floodplain_xns_qc, waterbody_mask, "WBD_Flag", output=Paths.floodplain_xns
    )